# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/bone_density/bone_density.parquet'
df_bone = read_parquet(pfile)
print(df_bone.shape)
df_bone.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_bone)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Data Filtering

In [ ]:
btest_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'hmo']
btest_metadata_col = [col for col in btest_metadata_col if col in df_bone.columns]
df_bone_metadata = df_bone[btest_metadata_col]
df_bone_features = df_bone.drop(columns=btest_metadata_col)

In [ ]:
missing_ratio_per_feature = df_bone_features.isnull().sum() / len(df_bone_features)  # 计算每列的缺失值比例
feature_threshold = 0.5  # 设定特征缺失率阈值

# 剔除缺失率超过阈值的特征
df_bone_filtered = df_bone_features.loc[:, missing_ratio_per_feature <= feature_threshold]
print(df_bone_filtered.shape)

# Imputation

In [ ]:
means = df_bone_filtered.mean()
df_bone_imputed = df_bone_filtered.fillna(means)

# Save

In [ ]:
df_bone_metadata.reset_index(drop=True, inplace=True)
df_bone_imputed.reset_index(drop=True, inplace=True)
df_bone = pd.concat([df_bone_metadata, df_bone_imputed], axis=1).dropna(axis=1)
pfile = 'pheno_data/202512/022-bone_density_imputed.parquet'
df_bone.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/022-bone_density_imputed.parquet'
df_bone = pd.read_parquet(pfile)
print(df_bone.shape)
df_bone.head()